# 18b · The Phase 4 threshold rule applied to the original recipe

After the 13-fold evidence (`17b`, section 4; `17c`), the project owner decided to use the **original recipe** (notebook 08) for the final system, not variant A. This notebook applies **the pre-declared Phase 4 choice rule, unchanged**, to the original recipe's 13 leave-one-prompt-out score sets (`results/16b_lopo`, seed 42), and uses whatever it picks. Nothing about the rules, targets, statistics, bootstrap or choice rule from `18_robust_threshold.ipynb` sections 1-6 is changed; the choice is not made between (b) and (d) on variant A's numbers. The held-out prompts and held-out generators are not scored here.

For reference, the declared rule (`18_robust_threshold.ipynb`, section 5): decide at the **1% target**; qualifying rules are those whose point estimate of mean AI caught is within **1.0 point of rule (a)**; choose the qualifying rule with the **smallest share of topics over target**, ties to the higher mean AI caught; if no rule other than (a) qualifies, or (a) is the choice, report the trade-off and ask.

New in this notebook, as the owner requested: **P(AI | essay in the cannot-tell band)**, at the dataset's AI share and at lower AI shares (below), and band wording if most essays in the band are human.

Part of the declaration for the P(AI | zone) calculation (written before computing it): for a zone with share $h$ of human essays and share $a$ of AI essays (leave-one-topic-out, equal weight per topic, as in the band table of notebook 18),
$$P(\mathrm{AI}\mid\text{zone}) = \frac{\pi\,a}{\pi\,a + (1-\pi)\,h},$$
with the AI prior $\pi$ = the dataset's AI share (34.6%, as in train, val and test), and also 10% and 5%. This is Bayes' rule and assumes the essays arriving later have the same human and AI shares in each zone as these 13 topics; the prior is the only thing that changes. Intervals: the same bootstrap as before.

In [1]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

from src.data import load_splits
from src.conformal import load_topics, compare_rules, band_report, set_tau, _p_arrays, loto, RULES

SEED, N_BOOT = 42, 1000
pd.set_option('display.width', 230); pd.set_option('display.max_colwidth', 100)
data = load_splits()
topics_O = load_topics('../results/16b_lopo', range(13), data)
print('original recipe, 13 left-out topics:', [t['name'][:16] for t in topics_O])

original recipe, 13 left-out topics: ['"A Cowboy Who Ro', 'Car-free cities', 'Cell phones at s', 'Community servic', 'Distance learnin', 'Driverless cars', 'Exploring Venus', 'Facial action co', 'Grades for extra', 'Mandatory extrac', 'Phones and drivi', 'Seeking multiple', 'The Face on Mars']


## 1 · The four rules, leave-one-topic-out, original recipe: the full table (1% and 0.5% targets)

In [2]:
cmp_O = compare_rules(topics_O, targets=(0.01, 0.005), n_boot=N_BOOT, seed=SEED)
cmp_O.round(4).to_csv('../results/18b_rule_comparison_original.csv', index=False)
def fmt(df):
    out = pd.DataFrame({'rule': df['rule'], 'target': df['target'].map('{:.1%}'.format)})
    for k, name in [('mean_fpr', 'mean false flags %'), ('worst_fpr', 'worst topic false flags %'), ('share_over_target', 'topics over target %'), ('mean_ai_caught', 'mean AI caught %')]:
        out[name] = [f"{r[k]:.2f} [{r[k + '_lo']:.2f}, {r[k + '_hi']:.2f}]" for _, r in df.iterrows()]
    return out.set_index(['target', 'rule'])
fmt(cmp_O)

,,mean false flags %,worst topic false flags %,topics over target %,mean AI caught %
target,rule,,,,
1.0%,a,"2.71 [2.27, 3.26]","5.24 [4.83, 9.22]","76.92 [61.54, 84.62]","99.46 [99.25, 99.70]"
0.5%,a,"1.32 [1.07, 1.71]","3.35 [2.71, 5.48]","69.23 [53.85, 84.62]","99.30 [98.91, 99.52]"
1.0%,b,"1.09 [1.00, 1.19]","3.22 [2.51, 4.90]","46.15 [30.77, 53.85]","99.16 [98.61, 99.41]"
0.5%,b,"0.53 [0.50, 0.67]","1.74 [1.30, 3.80]","46.15 [23.08, 53.85]","98.67 [97.81, 99.07]"
1.0%,c,"0.34 [0.10, 0.47]","1.21 [1.02, 2.96]","7.69 [7.69, 7.69]","98.12 [7.64, 98.78]"
0.5%,c,"0.17 [0.04, 0.24]","0.65 [0.51, 1.74]","7.69 [7.69, 7.69]","82.30 [7.55, 82.81]"
1.0%,d,"0.58 [0.44, 0.88]","1.74 [1.22, 4.12]","23.08 [23.08, 23.08]","98.78 [97.74, 99.21]"
0.5%,d,"0.25 [0.17, 0.50]","0.77 [0.66, 3.03]","23.08 [23.08, 23.08]","97.76 [30.39, 98.61]"


## 2 · The pre-declared choice, applied unchanged

In [3]:
c1 = cmp_O[cmp_O['target'] == 0.01].set_index('rule')
ref = c1.loc['a', 'mean_ai_caught']
qual = [r for r in RULES if c1.loc[r, 'mean_ai_caught'] >= ref - 1.0]
print(f"mean AI caught under rule (a): {ref:.2f}; qualifying rules (within 1.0 point): {qual}")
for r in RULES:
    print(f"   ({r}) mean AI caught {c1.loc[r, 'mean_ai_caught']:.3f}  ({c1.loc[r, 'mean_ai_caught'] - ref:+.3f} vs (a)), topics over target {c1.loc[r, 'share_over_target']:.1f}%")
best = sorted(qual, key=lambda r: (c1.loc[r, 'share_over_target'], -c1.loc[r, 'mean_ai_caught']))[0]
print(f"smallest share of topics over target among the qualifying rules: rule ({best}): {c1.loc[best, 'share_over_target']:.1f}% of topics over target")
if set(qual) == {'a'} or best == 'a':
    print('=> ONLY rule (a) qualifies or (a) is the choice: the declared procedure says report the trade-off and ASK.')
else:
    print(f'=> CHOSEN RULE for the original recipe: ({best})')
CHOSEN = best

mean AI caught under rule (a): 99.46; qualifying rules (within 1.0 point): ['a', 'b', 'd']
   (a) mean AI caught 99.464  (+0.000 vs (a)), topics over target 76.9%
   (b) mean AI caught 99.155  (-0.309 vs (a)), topics over target 46.2%
   (c) mean AI caught 98.122  (-1.342 vs (a)), topics over target 7.7%
   (d) mean AI caught 98.783  (-0.681 vs (a)), topics over target 23.1%
smallest share of topics over target among the qualifying rules: rule (d): 23.1% of topics over target
=> CHOSEN RULE for the original recipe: (d)


### The 13 topics one by one, at the 1% target

In [4]:
rows = []
pO = _p_arrays(topics_O)
for r in RULES:
    per = loto(pO, r, 0.01)
    for t, x in zip(topics_O, per.itertuples()):
        rows.append({'rule': r, 'topic': t['name'], 'false flags %': 100 * x.fpr, 'AI caught %': 100 * x.ai_caught})
bt = pd.DataFrame(rows)
bt.round(3).to_csv('../results/18b_by_topic_1pct_original.csv', index=False)
tab = bt.pivot(index='topic', columns='rule', values='false flags %'); tab['humans'] = [len(t['left_h']) for t in topics_O]
tab.round(2)

rule,a,b,c,d,humans
topic,,,,,
"""A Cowboy Who Rode the Waves""",3.57,1.17,0.66,0.80,1372
Car-free cities,1.31,0.23,0.04,0.04,2666
Cell phones at school,0.36,0.12,0.06,0.06,1656
Community service,4.99,1.43,0.39,1.04,1542
Distance learning,5.24,2.64,1.21,1.34,2157
Driverless cars,0.69,0.27,0.00,0.05,1886
Exploring Venus,4.67,1.56,0.27,0.54,1862
Facial action coding system,2.45,0.42,0.14,0.18,2167
Grades for extracurricular activities,1.54,0.55,0.18,0.31,1626


## 3 · The cannot-tell band with the chosen rule (edges as declared: flag at the rule's 1% threshold, cannot tell up to its 5% threshold)

In [5]:
band = band_report(topics_O, CHOSEN, 0.01, 0.05, n_boot=N_BOOT, seed=SEED)
band.round(2).to_csv('../results/18b_band_original.csv')
print(f'rule ({CHOSEN}), original recipe, leave-one-topic-out over 13 topics')
band.round(2)

rule (d), original recipe, leave-one-topic-out over 13 topics


,estimate,lo,hi
human flagged %,0.58,0.44,0.88
human in band %,2.49,1.84,2.98
human not flagged %,96.93,96.42,97.45
AI flagged %,98.78,97.74,99.21
AI in band %,0.69,0.39,1.69
AI missed %,0.53,0.26,0.75
topics over target (flagged),23.08,23.08,23.08


In [6]:
sens = {f'{a:.1%} to {b:.0%}': band_report(topics_O, CHOSEN, a, b, n_boot=200, seed=SEED)['estimate'] for a, b in [(0.005, 0.05), (0.01, 0.10)]}
pd.DataFrame(sens).round(2)

,0.5% to 5%,1.0% to 10%
human flagged %,0.25,0.58
human in band %,2.82,5.54
human not flagged %,96.93,93.88
AI flagged %,97.76,98.78
AI in band %,1.71,0.99
AI missed %,0.53,0.23
topics over target (flagged),23.08,23.08


## 4 · P(AI | zone): how much of each zone is AI, at different AI shares

Zones: **flagged** ($p \le \tau_{1\%}$), **cannot tell** ($\tau_{1\%} < p \le \tau_{5\%}$), **not flagged** ($p > \tau_{5\%}$), from the chosen rule, leave-one-topic-out. Priors: the dataset's AI share (34.6%), 10% and 5%.

In [7]:
PRIORS = {'34.6% (the dataset)': 0.346, '10%': 0.10, '5%': 0.05}
def zone_shares(per):
    h = {'flagged': per['fpr'].mean(), 'cannot tell': per['human_in_band'].mean(), 'not flagged': per['human_not_flagged'].mean()}
    a = {'flagged': per['ai_caught'].mean(), 'cannot tell': per['ai_in_band'].mean(), 'not flagged': per['ai_missed'].mean()}
    return h, a
def p_ai(h, a, prior):
    return {z: prior * a[z] / (prior * a[z] + (1 - prior) * h[z]) if (prior * a[z] + (1 - prior) * h[z]) > 0 else np.nan for z in h}
rng = np.random.default_rng(SEED)
draws = {(z, k): [] for z in ('flagged', 'cannot tell', 'not flagged') for k in PRIORS}
for _ in range(N_BOOT):
    per = loto(_p_arrays(topics_O, rng), CHOSEN, 0.01, 0.05)
    h, a = zone_shares(per)
    for k, pi in PRIORS.items():
        for z, v in p_ai(h, a, pi).items(): draws[(z, k)].append(v)
h0, a0 = zone_shares(loto(pO, CHOSEN, 0.01, 0.05))
rows = []
for k, pi in PRIORS.items():
    pt = p_ai(h0, a0, pi)
    for z in ('flagged', 'cannot tell', 'not flagged'):
        d = np.array(draws[(z, k)], dtype=float); d = d[~np.isnan(d)]
        rows.append({'AI share of arriving essays': k, 'zone': z, 'P(AI | zone) %': f'{100 * pt[z]:.1f} [{100 * np.percentile(d, 2.5):.1f}, {100 * np.percentile(d, 97.5):.1f}]'})
pai = pd.DataFrame(rows).pivot(index='zone', columns='AI share of arriving essays', values='P(AI | zone) %').loc[['flagged', 'cannot tell', 'not flagged']][list(PRIORS)]
pai.to_csv('../results/18b_p_ai_by_zone_original.csv')
print('share of humans per zone:', {z: f'{100 * v:.2f}%' for z, v in h0.items()}, '| share of AI per zone:', {z: f'{100 * v:.2f}%' for z, v in a0.items()})
pai

share of humans per zone: {'flagged': '0.58%', 'cannot tell': '2.49%', 'not flagged': '96.93%'} | share of AI per zone: {'flagged': '98.78%', 'cannot tell': '0.69%', 'not flagged': '0.53%'}


AI share of arriving essays,34.6% (the dataset),10%,5%
zone,,,
flagged,"98.9 [98.3, 99.2]","95.0 [92.6, 96.2]","90.0 [85.6, 92.2]"
cannot tell,"12.8 [8.0, 27.4]","3.0 [1.8, 7.3]","1.4 [0.9, 3.6]"
not flagged,"0.3 [0.1, 0.4]","0.1 [0.0, 0.1]","0.0 [0.0, 0.0]"


## 5 · The thresholds for Phase 5, on the p scale, from all 13 topics (no topic left out)

In [8]:
all_ph = [p[0] for p in _p_arrays(topics_O)]
final = pd.DataFrame({f'{t:.1%}': {r: set_tau(r, all_ph, t) for r in RULES} for t in (0.005, 0.01, 0.05)})
final.index.name = 'rule'
final.round(5).to_csv('../results/18b_final_thresholds_p_scale_original.csv')
print(f'tau on the p scale from all 13 topics, original recipe. Chosen rule: ({CHOSEN}).')
final.round(5)

tau on the p scale from all 13 topics, original recipe. Chosen rule: (d).


,0.5%,1.0%,5.0%
rule,,,
a,0.00500,0.01000,0.05000
b,0.00189,0.00378,0.01976
c,0.00048,0.00095,0.00951
d,0.00093,0.00235,0.01032


## Verdict

*(Written after running the cells above. The rules, targets, statistics, bootstrap and the choice rule are those of notebook 18, applied unchanged; the P(AI | zone) definition was committed before it was computed, commit 849cb09.)*

**1. The declared rule picks rule (d), the topic quantile, for the original recipe.** Qualifying rules (mean AI caught within 1.0 point of rule (a)): (a), (b) and (d); rule (c) is out (98.12%, 1.34 points below). Among the qualifying rules (d) has the smallest share of topics over target. Here (d) qualifies, with 0.68 points to spare, unlike on variant A, where it missed by 0.01 point.

| rule, 1% target | mean false flags % | worst topic % | topics over target % | mean AI caught % |
|---|---|---|---|---|
| (a) pooled val | 2.71 [2.27, 3.26] | 5.24 [4.83, 9.22] | 76.9 [61.5, 84.6] | 99.46 [99.25, 99.70] |
| (b) pooled LOPO | 1.09 [1.00, 1.19] | 3.22 [2.51, 4.90] | 46.2 [30.8, 53.8] | 99.16 [98.61, 99.41] |
| (c) worst topic | 0.34 [0.10, 0.47] | 1.21 [1.02, 2.96] | 7.7 [7.7, 7.7] | 98.12 [7.64, 98.78] |
| **(d) topic quantile, chosen** | **0.58 [0.44, 0.88]** | **1.74 [1.22, 4.12]** | **23.1 [23.1, 23.1]** | **98.78 [97.74, 99.21]** |

| rule, 0.5% target | mean false flags % | worst topic % | topics over target % | mean AI caught % |
|---|---|---|---|---|
| (a) | 1.32 [1.07, 1.71] | 3.35 [2.71, 5.48] | 69.2 [53.8, 84.6] | 99.30 [98.91, 99.52] |
| (b) | 0.53 [0.50, 0.67] | 1.74 [1.30, 3.80] | 46.2 [23.1, 53.8] | 98.67 [97.81, 99.07] |
| (c) | 0.17 [0.04, 0.24] | 0.65 [0.51, 1.74] | 7.7 [7.7, 7.7] | 82.30 [7.55, 82.81] |
| **(d)** | **0.25 [0.17, 0.50]** | **0.77 [0.66, 3.03]** | **23.1 [23.1, 23.1]** | **97.76 [30.39, 98.61]** |

- **Rule (d) against the status quo (a):** mean false flags on a topic never seen fall from 2.71% to 0.58%, the worst topic from 5.24% to 1.74%, and topics over the 1% target from 77% to 23% (Community service 1.04%, Distance learning 1.34%, Seeking multiple opinions 1.74%), at a cost of 0.68 points of AI caught.
- **Caution on the (d) AI-caught interval at the 0.5% target:** its lower end (30%) comes from bootstrap resamples in which the 20th-percentile topic threshold collapses; the point estimate is 97.76%. The 1% target is more stable (lower end 97.74%).
- **Final thresholds on the p scale from all 13 topics, rule (d)** (`results/18b_final_thresholds_p_scale_original.csv`): **tau(0.5%) = 0.00093, tau(1%) = 0.00235, tau(5%) = 0.01032**. In words: for the "1%" operating point, flag only essays that score above all but 0.235% of the val humans of that model, and the cannot-tell band ends at the top 1.03%. (For reference, rule (b): 0.00189, 0.00378, 0.01976; rule (a): the nominal 0.005, 0.01, 0.05.)
- **Compared with variant A under its chosen rule (b)** (notebook 18: 1.58% mean, 12.61% worst topic, 31% over target, 98.97% AI caught), the original recipe with rule (d) is better on every false-flag statistic and about equal on detection.

**2. The cannot-tell band is overwhelmingly human, so it must not be worded as a 50/50 call.** Band edges as declared (flag at tau(1%), cannot tell up to tau(5%), rule (d)), leave-one-topic-out over 13 topics:

| zone | human essays | AI essays |
|---|---|---|
| flagged | 0.58% [0.44, 0.88] | 98.78% [97.74, 99.21] |
| **cannot tell** | **2.49% [1.84, 2.98]** | **0.69% [0.39, 1.69]** |
| not flagged | 96.93% [96.42, 97.45] | 0.53% [0.26, 0.75] |

How much of each zone is AI depends on how many of the essays arriving are AI-written (Bayes' rule, as declared), P(AI | zone):

| zone | AI share of arriving essays 34.6% (the dataset) | 10% | 5% |
|---|---|---|---|
| flagged | 98.9% [98.3, 99.2] | 95.0% [92.6, 96.2] | 90.0% [85.6, 92.2] |
| **cannot tell** | **12.8% [8.0, 27.4]** | **3.0% [1.8, 7.3]** | **1.4% [0.9, 3.6]** |
| not flagged | 0.3% [0.1, 0.4] | 0.1% | 0.0% |

- **Most essays in the band are human, at every AI share we looked at:** about 7 in 8 at the dataset's one-third AI share, about 32 in 33 at 10% and about 70 in 71 at 5%. A label such as "cannot tell" is easy to read as "could be either, 50/50", which these numbers contradict.
- **Even the flagged zone is not proof at low AI shares:** at 10% AI essays about 1 in 20 flagged essays is a student's, and at 5% about 1 in 10. This follows from the false-flag rate being small but non-zero on 90-95% of the essays, and it bears on the project's own wording ("a reason to ask questions, not evidence").
- **In natural frequencies,** in a class of 1,000 essays of which 100 (10%) are AI-written: about 99 of the 100 AI essays are flagged, 0.7 are in the band, 0.5 are missed; of the 900 student essays, about 5 are flagged, **22** are in the band and 872 are not flagged. So of the roughly 104 flagged essays, about 5 are students'; of the roughly 23 essays in the band, about 22 are students'.

**3. Proposed wording and edges (not applied; the front end is untouched).**
- **Name and say what the zone is.** Instead of "Cannot tell", something like **"Unusual, but common for student essays"**, with a plain sentence such as *"Essays in this range are mostly written by students (in our tests, about 7 in 8 when a third of essays are AI-written, and about 32 in 33 when 1 in 10 is). This is not evidence of AI writing."* The prior is the one thing the number depends on, so state it or give natural frequencies for a stated class mix.
- **Keep the flagged wording honest about the base rate:** *"This reads like the AI essays we tested. If 1 in 10 essays in a class is AI-written, about 1 in 20 flagged essays is still a student's."*
- **Or drop the middle label** and show two outcomes (flagged / not flagged), keeping the band's calculation only as an internal "borderline" note: the band holds 2.5% of student essays and 0.7% of AI essays, so it adds little power to detection and mainly protects students.
- **Edges.** The band exists to stop students on the shifted tail being flagged; for that, the declared edges (1% to 5% of the rule's thresholds) are reasonable. Narrower edges (for example 1% to 2%) would shrink the human share in the band without changing what the AI share in it looks like (almost no AI essays land there), so a different edge cannot make the band a 50/50 call; it can only make it smaller. If a 50/50 band is wanted, it would have to sit nearer the flag line, where it would contain very few essays of either kind.

**Limits.** Single-seed folds, 13 topics; the intervals exclude seed variation (seed-to-seed variation of a fold's rate was about 0.3-0.7 points, notebook 16b); the zone shares weight each topic equally and assume future essays have the same shares in each zone as these 13 topics; Bayes' rule with a stated AI prior assumes that prior is right for the setting. The held-out prompts and generators were not scored.

## Key takeaways

- **The declared rule picks (d) for the original recipe:** mean false flags 0.58% (against 2.71% for pooled-val thresholds), worst topic 1.74%, 23% of topics over target, AI caught 98.78%.
- **Thresholds on the p scale for the final system (rule (d), from all 13 topics):** tau(0.5%) = 0.00093, tau(1%) = 0.00235, tau(5%) = 0.01032.
- **The cannot-tell band holds 2.5% of student essays and 0.7% of AI essays and is mostly human** (about 87% human at one-third AI, 97% at 10%, 98.6% at 5%): word it as "unusual but common for student essays", not as a coin flip.
- **A flag is not proof when AI essays are rare:** at 10% AI, about 1 flagged essay in 20 is a student's.